In [ ]:
%pip install yfinance

In [1]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import os

# 1. Setup
yesterday = (datetime.now() - timedelta(1)).strftime('%Y-%m-%d')
start_date = "2010-01-01"

# 2. TICKERS
base_lkr = "USDLKR=X" 
minor_vs_usd = {
    "EURUSD=X": "EUR",
    "GBPUSD=X": "GBP",
    "AUDUSD=X": "AUD",
    "CADUSD=X": "CAD",
    "JPY=X": "JPY"  # Using the more stable JPY ticker
}
macro_tickers = {"DX-Y.NYB": "USD_Index"}

print(f"Initializing Resilient Data Collection...")

# 3. DOWNLOAD
all_tickers = [base_lkr] + list(minor_vs_usd.keys()) + list(macro_tickers.keys())

try:
    # Download and select 'Close'
    raw_data = yf.download(all_tickers, start=start_date, end=yesterday)['Close']
    
    # --- THE FIX FOR THE 'DATE' AMBIGUITY ---
    # We explicitly reset the index and rename it to 'Date_Index' to avoid conflicts
    raw_data = raw_data.reset_index()
    raw_data.rename(columns={'Date': 'Date_Index'}, inplace=True)
    
    # Clean weekend/holiday gaps
    raw_data = raw_data.ffill().bfill() 

    # Prepare macro features
    macro_df = raw_data[['Date_Index', 'DX-Y.NYB']].copy()
    macro_df.rename(columns={'DX-Y.NYB': 'USD_Index'}, inplace=True)

    # 4. TRIANGULATION MATH
    data_list = []
    
    # A. USD (Direct)
    usd_df = pd.DataFrame({
        'Date': raw_data['Date_Index'], 
        'LKR_Rate': raw_data[base_lkr], 
        'Currency': 'USD'
    })
    data_list.append(usd_df.merge(macro_df, left_on='Date', right_on='Date_Index').drop(columns=['Date_Index']))

    # B. Minors (Rate * USDLKR)
    for t_id, name in minor_vs_usd.items():
        if t_id in raw_data.columns:
            curr_df = pd.DataFrame({
                'Date': raw_data['Date_Index'], 
                'LKR_Rate': raw_data[t_id] * raw_data[base_lkr], 
                'Currency': name
            })
            data_list.append(curr_df.merge(macro_df, left_on='Date', right_on='Date_Index').drop(columns=['Date_Index']))

    # 5. CONSOLIDATE & SAVE
    master_df = pd.concat(data_list, ignore_index=True)
    
    output_path = "../data/raw/LKR_Forex_Macro_Raw.csv"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    master_df.to_csv(output_path, index=False)

    print(f"\nSUCCESS: Master data saved to {output_path}")
    print(f"Total Rows: {len(master_df)}")
    print(f"Currencies: {master_df['Currency'].unique()}")

except Exception as e:
    print(f"Critical Error during download: {e}")

Initializing Resilient Data Collection...


[*********************100%***********************]  7 of 7 completed



SUCCESS: Master data saved to ../data/raw/LKR_Forex_Macro_Raw.csv
Total Rows: 25224
Currencies: ['USD' 'EUR' 'GBP' 'AUD' 'CAD' 'JPY']
